In [202]:
# Verification cell: just run it (Shift+Enter). Provided for you.
# importlib.util is a submodule: importing the parent alone does not bind it,
# and this cell now runs before anything else has imported it for us.
import importlib.util

REQUIRED = [
    "numpy", "pandas", "matplotlib", "seaborn", "sklearn", "openpyxl",
]
missing = [name for name in REQUIRED if importlib.util.find_spec(name) is None]

if missing:
    print("MISSING packages:", ", ".join(missing))
    print("-> Please install the missing packages using pip install <package-name>")
else:
    print("Environment OK.\nEvery package the project needs is installed!")

Environment OK.
Every package the project needs is installed!


In [203]:
# Import all packages needed for the project
# Unused packages can be safely removed from this cell
import sys
from functools import partial
from pathlib import Path

if str(Path.cwd()) not in sys.path:
    sys.path.insert(0, str(Path.cwd()))

import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.tree import DecisionTreeClassifier
from sklearn.metrics import confusion_matrix, f1_score
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import OrdinalEncoder
from sklearn.pipeline import make_pipeline
# import matplotlib.pyplot as plt
# import seaborn as sns
# import openpyxl as op

RANDOM_STATE = 42
np.random.seed(RANDOM_STATE)
FIGSIZE = (10, 6)



In [204]:
# Import data that is gonna be used for this project

RAW_TRAIN_DATA_PATH = "../data/train.csv"
CLIENT_DATA_PATH = "../data/clientes.csv"
raw_train_data = pd.read_csv(RAW_TRAIN_DATA_PATH, index_col="ID", float_precision='round_trip')
client_data = pd.read_csv(CLIENT_DATA_PATH, index_col="customer_id", float_precision='round_trip')

In [205]:
print(f"Raw train data: {raw_train_data.shape[0]:,} rows x {raw_train_data.shape[1]} feature columns, indexed by ID")
raw_train_data.head()

Raw train data: 6,534 rows x 38 feature columns, indexed by ID


,Total a Pagar,Total Bruto,Total Liquido,Total Imp_,Total Portes,Total Desc_ Global,Total Desc_ Linha,Custo,N_ Detalhes,invoice_count_current_purchase,...,spend_365d,customer_id,payment_type,salesperson,commercial_zone,transport_method,warehouse,document_series,purchase_date,random_noise
ID,,,,,,,,,,,,,,,,,,,,,
10000000,558.11,510.90,453.75,104.36,0.0,0.0,57.15,153.099911,4.0,1.0,...,858.30,660784,Maybe Tomorrow Payment 10,VLOOKUP Master 9,Decorative Dashboard Zone 4,Maybe Tomorrow Delivery 61,Final Excel Warehouse 1,Ctrl Z Series 2,2024-10-29,-0.660448
10000001,978.80,830.02,795.77,183.03,0.0,0.0,34.25,472.943134,32.0,1.0,...,2500.68,189146,Maybe Tomorrow Payment 7,VLOOKUP Master 1,Decorative Dashboard Zone 4,Maybe Tomorrow Delivery 36,Final Excel Warehouse 1,Ctrl Z Series 1,2017-07-08,0.924510
10000002,10.33,12.00,8.40,1.93,0.0,0.0,3.60,6.097485,1.0,1.0,...,7899.80,103940,Maybe Tomorrow Payment 9,VLOOKUP Master 1,Decorative Dashboard Zone 4,NaN,Final Excel Warehouse 1,Ctrl Z Series 3,2019-03-19,-0.258730
10000003,856.83,696.61,696.61,160.22,0.0,0.0,0.00,408.167275,9.0,1.0,...,270.33,512927,Maybe Tomorrow Payment 7,VLOOKUP Master 4,Decorative Dashboard Zone 4,Maybe Tomorrow Delivery 71,Final Excel Warehouse 1,Ctrl Z Series 3,2018-11-10,1.448402
10000004,1616.10,1488.85,1313.90,302.20,0.0,0.0,174.95,485.401864,33.0,1.0,...,2062.50,122356,Maybe Tomorrow Payment 12,NaN,Decorative Dashboard Zone 4,Maybe Tomorrow Delivery 61,Final Excel Warehouse 1,Ctrl Z Series 2,2021-11-30,-1.058890


In [206]:
print(f"Clients data: {client_data.shape[0]:,} rows x {client_data.shape[1]} feature columns, indexed by ID")
client_data.head()

Clients data: 1,143 rows x 12 feature columns, indexed by ID


,country_or_region,commercial_region,island,municipality,city,postal_area,postal_code,street,address,customer_type,company_name,customer_since_year
customer_id,,,,,,,,,,,,
842570,Country CT11: Land of The Final Version,Region RG37: We Will See Later,NaN,NaN,City CY417: Missing Cell Port,NaN,Postal code PC828: #REF! Delivery Route,NaN,Address AD1120: Cell Without a Home,NaN,Excel Is Not A Database Ltd 150,2018
770941,Country CT9: Land of The Final Version,NaN,NaN,NaN,NaN,NaN,Postal code PC56: #REF! Delivery Route,NaN,Address AD62: Cell Without a Home,NaN,Excel Is Not A Database Ltd 1064,2021
827053,Country CT10: Land of The Final Version,Region RG17: We Will See Later,Island IS2: The Pivot Archipelago,Municipality MU14: Council of Last Minute Edits,NaN,Postal area PA271: Hidden Sheet Sector,Postal code PC827: #REF! Delivery Route,Street ST170: PowerPoint New Walk,Address AD449: Cell Without a Home,NaN,Excel Is Not A Database Ltd 113,2016
494648,Country CT10: Land of The Final Version,Region RG17: We Will See Later,Island IS2: The Pivot Archipelago,Municipality MU14: Council of Last Minute Edits,NaN,Postal area PA271: Hidden Sheet Sector,Postal code PC826: #REF! Delivery Route,NaN,Address AD454: Cell Without a Home,NaN,Excel Is Not A Database Ltd 430,2015
429291,Country CT10: Land of The Final Version,Region RG17: We Will See Later,Island IS2: The Pivot Archipelago,Municipality MU14: Council of Last Minute Edits,NaN,Postal area PA271: Hidden Sheet Sector,Postal code PC826: #REF! Delivery Route,NaN,Address AD453: Cell Without a Home,NaN,Excel Is Not A Database Ltd 673,2015


In [207]:
def out_of_domain(frame, cols, errors):
    """Return rows with at least one known domain violation in `cols`."""
    bad = pd.Series(False, index=frame.index)
    for column in cols:
        bad |= errors[column](frame[column])
    return bad


# These rules describe impossible values for purchase records; missing values
# are reported separately because they may need imputation rather than removal.
train_data_domain_errors = {
    "Total a Pagar": lambda values: values.notna() & (values <= 0),
    "Total Bruto": lambda values: values.notna() & (values <= 0),
    "Total Liquido": lambda values: values.notna() & (values < 0),
    "Total Imp_": lambda values: values.notna() & (values < 0),
    "Total Portes": lambda values: values.notna() & (values < 0),
    "Total Desc_ Global": lambda values: values.notna() & (values < 0),
    "Total Desc_ Linha": lambda values: values.notna() & (values < 0),
    "Custo": lambda values: values.notna() & (values < 0),
    "N_ Detalhes": lambda values: values.notna() & ((values < 1) | (values % 1 != 0)),
    "invoice_count_current_purchase": lambda values: values.notna() & ((values < 1) | (values % 1 != 0)),
    "repeat_purchase_90d": lambda values: values.notna() & ~values.isin([0, 1]),
    "days_since_previous_purchase": lambda values: values.notna() & (values < 0),
    "customer_tenure_days": lambda values: values.notna() & (values < 0),
    "prior_purchase_count": lambda values: values.notna() & ((values < 0) | (values % 1 != 0)),
    "prior_total_spend": lambda values: values.notna() & (values < 0),
    "prior_mean_purchase_value": lambda values: values.notna() & (values < 0),
    "prior_std_purchase_value": lambda values: values.notna() & (values < 0),
    "prior_min_purchase_value": lambda values: values.notna() & (values < 0),
    "prior_max_purchase_value": lambda values: values.notna() & (values < 0),
    "prior_mean_gap_days": lambda values: values.notna() & (values < 0),
    "prior_std_gap_days": lambda values: values.notna() & (values < 0),
    "purchase_count_30d": lambda values: values.notna() & ((values < 0) | (values % 1 != 0)),
    "spend_30d": lambda values: values.notna() & (values < 0),
    "purchase_count_90d": lambda values: values.notna() & ((values < 0) | (values % 1 != 0)),
    "spend_90d": lambda values: values.notna() & (values < 0),
    "purchase_count_180d": lambda values: values.notna() & ((values < 0) | (values % 1 != 0)),
    "spend_180d": lambda values: values.notna() & (values < 0),
    "purchase_count_365d": lambda values: values.notna() & ((values < 0) | (values % 1 != 0)),
    "spend_365d": lambda values: values.notna() & (values < 0),
    "customer_id": lambda values: values.notna() & ((values <= 0) | (values % 1 != 0)),
}

numeric_columns = set(train_data_domain_errors) | {"random_noise"}
column_report = []
invalid_by_column = {}
for column in raw_train_data.columns:
    values = raw_train_data[column]
    invalid = pd.Series(False, index=raw_train_data.index)

    if column in numeric_columns:
        if pd.api.types.is_numeric_dtype(values):
            invalid |= values.notna() & ~np.isfinite(values)
            if column in train_data_domain_errors:
                invalid |= train_data_domain_errors[column](values)
        else:
            invalid |= values.notna()
    elif column == "purchase_date":
        parsed_dates = pd.to_datetime(values, errors="coerce")
        invalid |= values.notna() & parsed_dates.isna()
    else:
        invalid |= values.notna() & ~values.map(lambda value: isinstance(value, str))
        invalid |= values.map(lambda value: isinstance(value, str) and not value.strip())

    invalid_by_column[column] = invalid
    examples = [
        f"ID {row_id}: {value!r}"
        for row_id, value in values[invalid].head(3).items()
    ]
    column_report.append({
        "column": column,
        "missing": int(values.isna().sum()),
        "invalid": int(invalid.sum()),
        "invalid_examples": "; ".join(examples),
    })

quality_report = pd.DataFrame(column_report).set_index("column")
invalid_rows = out_of_domain(raw_train_data, list(train_data_domain_errors), train_data_domain_errors)
for invalid in invalid_by_column.values():
    invalid_rows |= invalid
print(f"Checked {len(raw_train_data.columns)} columns across {len(raw_train_data):,} rows.")
row_ids = pd.to_numeric(pd.Series(raw_train_data.index), errors="coerce")
invalid_ids = row_ids.isna() | (row_ids <= 0) | (row_ids % 1 != 0)
print(f"Rows with at least one invalid value: {invalid_rows.sum():,}")
print(f"Invalid row IDs: {invalid_ids.sum():,}; duplicate row IDs: {raw_train_data.index.has_duplicates}")
display(quality_report)

Checked 38 columns across 6,534 rows.
Rows with at least one invalid value: 63
Invalid row IDs: 0; duplicate row IDs: False


,missing,invalid,invalid_examples
column,,,
Total a Pagar,65,0,
Total Bruto,65,0,
Total Liquido,65,0,
Total Imp_,65,0,
Total Portes,65,0,
Total Desc_ Global,65,0,
Total Desc_ Linha,65,0,
Custo,65,0,
N_ Detalhes,65,0,


In [208]:
# Drop only rows identified as invalid by the checks above; retain missing values.
train_processed = raw_train_data.loc[~invalid_rows].copy()

processed_dir = Path("../data/processed")
processed_dir.mkdir(parents=True, exist_ok=True)
processed_path = processed_dir / "train_processed.csv"
train_processed.to_csv(processed_path, index=True, index_label=raw_train_data.index.name)

print(f"Removed {invalid_rows.sum():,} invalid rows; retained {len(train_processed):,} of {len(raw_train_data):,} rows.")
print(f"Saved processed training data to {processed_path.resolve()}")

Removed 63 invalid rows; retained 6,471 of 6,534 rows.
Saved processed training data to /home/yoru/Documents/novaims/Y1/S1/ML/ML_Project/data/processed/train_processed.csv


In [209]:
TRAIN_DATA_PATH = "../data/processed/train_processed.csv"

train_data = pd.read_csv(TRAIN_DATA_PATH, index_col='ID', float_precision='round_trip')
print(train_data.dtypes)

Total a Pagar                     float64
Total Bruto                       float64
Total Liquido                     float64
Total Imp_                        float64
Total Portes                      float64
Total Desc_ Global                float64
Total Desc_ Linha                 float64
Custo                             float64
N_ Detalhes                       float64
invoice_count_current_purchase    float64
repeat_purchase_90d                 int64
days_since_previous_purchase      float64
customer_tenure_days              float64
prior_purchase_count              float64
prior_total_spend                 float64
prior_mean_purchase_value         float64
prior_std_purchase_value          float64
prior_min_purchase_value          float64
prior_max_purchase_value          float64
prior_mean_gap_days               float64
prior_std_gap_days                float64
purchase_count_30d                float64
spend_30d                         float64
purchase_count_90d                

In [210]:
train_data.info()

<class 'pandas.DataFrame'>
Index: 6471 entries, 10000000 to 10007674
Data columns (total 38 columns):
 #   Column                          Non-Null Count  Dtype  
---  ------                          --------------  -----  
 0   Total a Pagar                   6406 non-null   float64
 1   Total Bruto                     6406 non-null   float64
 2   Total Liquido                   6406 non-null   float64
 3   Total Imp_                      6406 non-null   float64
 4   Total Portes                    6406 non-null   float64
 5   Total Desc_ Global              6406 non-null   float64
 6   Total Desc_ Linha               6406 non-null   float64
 7   Custo                           6406 non-null   float64
 8   N_ Detalhes                     6406 non-null   float64
 9   invoice_count_current_purchase  6406 non-null   float64
 10  repeat_purchase_90d             6471 non-null   int64  
 11  days_since_previous_purchase    5613 non-null   float64
 12  customer_tenure_days            6406 no

In [211]:
train_data.describe()

,Total a Pagar,Total Bruto,Total Liquido,Total Imp_,Total Portes,Total Desc_ Global,Total Desc_ Linha,Custo,N_ Detalhes,invoice_count_current_purchase,...,purchase_count_30d,spend_30d,purchase_count_90d,spend_90d,purchase_count_180d,spend_180d,purchase_count_365d,spend_365d,customer_id,random_noise
count,6406.000000,6406.000000,6406.000000,6406.000000,6406.000000,6406.000000,6406.000000,6406.000000,6406.000000,6406.000000,...,6406.000000,6406.000000,6406.000000,6406.000000,6406.000000,6406.000000,6406.000000,6.214000e+03,6471.000000,6406.000000
mean,926.272665,886.606792,773.892384,152.380281,1.157114,0.353348,112.361060,328.547360,21.518108,1.065720,...,0.597721,456.770292,1.628629,1254.386970,3.364658,2650.936207,6.539494,2.419669e+06,506344.863700,-0.022854
std,1336.671974,1327.845213,1137.351962,225.721171,5.870148,5.141547,289.686073,520.623700,27.725653,0.309448,...,1.207078,1192.675702,2.968045,2352.699744,5.433297,4027.662937,10.281514,4.907583e+07,266294.482345,0.998788
min,0.930000,0.830000,0.760000,0.000000,0.000000,0.000000,0.000000,0.000000,1.000000,1.000000,...,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000e+00,103940.000000,-3.529210
25%,188.905000,175.742500,154.245000,29.970000,0.000000,0.000000,0.000000,49.052750,4.000000,1.000000,...,0.000000,0.000000,0.000000,0.000000,1.000000,146.977500,1.000000,6.404950e+02,258332.000000,-0.677733
50%,476.580000,445.800000,389.710000,79.750000,0.000000,0.000000,13.225000,152.509906,11.000000,1.000000,...,0.000000,0.000000,1.000000,198.470000,2.000000,1117.240000,3.000000,2.438500e+03,489220.000000,-0.046120
75%,1087.940000,1038.140000,903.587500,184.932500,0.000000,0.000000,92.227500,396.527171,29.000000,1.000000,...,1.000000,336.932500,2.000000,1513.387500,3.000000,3401.027500,7.000000,7.053812e+03,755628.000000,0.642024
max,22297.210000,21578.280000,18127.810000,4169.400000,181.060000,218.490000,4103.440000,7427.400583,423.000000,5.000000,...,11.000000,22297.210000,22.000000,30234.880000,37.000000,49746.970000,60.000000,1.000000e+09,998870.000000,4.025824


In [212]:
# Check for imbalance between values in the columns 'repeat_purchase_90d'

train_data["repeat_purchase_90d"].value_counts(normalize=True)

repeat_purchase_90d
1    0.560037
0    0.439963
Name: proportion, dtype: float64

In [213]:
X = train_data.drop(columns='repeat_purchase_90d')
y = train_data['repeat_purchase_90d']

In [214]:
X_train, X_val, y_train, y_val = train_test_split(
    X,
    y,
    train_size=0.7,
    shuffle=True,
    stratify=y,
    random_state=RANDOM_STATE,
)
len(X_train), len(X_val)

(4529, 1942)

In [215]:
X_train.dtypes.value_counts()

float64    29
str         7
int64       1
Name: count, dtype: int64

In [216]:
# Separate categorical features from numeric features

categorical_cols = X_train.select_dtypes(include=["str"]).columns
numeric_cols = X_train.columns.difference(categorical_cols)

In [217]:
# Encode categories and leave numeric features unchanged

preprocessor = ColumnTransformer(
    transformers=[
        ("categorical", OrdinalEncoder(
            handle_unknown="use_encoded_value",
            unknown_value=-1
        ), categorical_cols),
        ("numeric", "passthrough", numeric_cols)
    ]
)

In [218]:
# Build and train the decision tree pipeline

tree = make_pipeline(
    preprocessor,
    DecisionTreeClassifier(random_state=RANDOM_STATE)
)
tree.fit(X_train, y_train)

,"steps steps: list of tuplesList of (name of step, estimator) tuples that are to be chained insequential order. To be compatible with the scikit-learn API, all stepsmust define `fit`. All non-last steps must also define `transform`. See:ref:`Combining Estimators <combining_estimators>` for more details.","[('columntransformer', ...), ('decisiontreeclassifier', ...)]"
,"transform_input transform_input: list of str, default=NoneThe names of the :term:`metadata` parameters that should be transformed by thepipeline before passing it to the step consuming it.This enables transforming some input arguments to ``fit`` (other than ``X``)to be transformed by the steps of the pipeline up to the step which requiresthem. Requirement is defined via :ref:`metadata routing <metadata_routing>`.For instance, this can be used to pass a validation set through the pipeline.You can only set this if metadata routing is enabled, which youcan enable using ``sklearn.set_config(enable_metadata_routing=True)``... versionadded:: 1.6",None
,"memory memory: str or object with the joblib.Memory interface, default=NoneUsed to cache the fitted transformers of the pipeline. The last stepwill never be cached, even if it is a transformer. By default, nocaching is performed. If a string is given, it is the path to thecaching directory. Enabling caching triggers a clone of the transformersbefore fitting. Therefore, the transformer instance given to thepipeline cannot be inspected directly. Use the attribute ``named_steps``or ``steps`` to inspect estimators within the pipeline. Caching thetransformers is advantageous when fitting is time consuming. See:ref:`sphx_glr_auto_examples_neighbors_plot_caching_nearest_neighbors.py`for an example on how to enable caching.",None
,"verbose verbose: bool, default=FalseIf True, the time elapsed while fitting each step will be printed as itis completed.",False
Name,Type,Value
"classes_ classes_: ndarray of shape (n_classes,)The classes labels. Only exist if the last step of the pipeline is aclassifier.","ndarray[int64](2,)","[0,1]"
"feature_names_in_ feature_names_in_: ndarray of shape (`n_features_in_`,)Names of features seen during :term:`fit`. Only defined if theunderlying estimator exposes such an attribute when fit... versionadded:: 1.0","ndarray[object](37,)","['Total a Pagar','Total Bruto','Total Liquido',...,'document_series', 'purchase_date','random_noise']"
n_features_in_ n_features_in_: intNumber of features seen during :term:`fit`. Only defined if theunderlying first estimator in `steps` exposes such an attributewhen fit... versionadded:: 0.24,int,37
,"transformers transformers: list of tuplesList of (name, transformer, columns) tuples specifying thetransformer objects to be applied to subsets of the data.name : str Like in Pipeline and FeatureUnion, this allows the transformer and its parameters to be set using ``set_params`` and searched in grid search.transformer : {'drop', 'passthrough'} or estimator Estimator must support :term:`fit` and :term:`transform`. Special-cased strings 'drop' and 'passthrough' are accepted as well, to indicate to drop the columns or to pass them through untransformed, respectively.columns : str, array-like of str, int, array-like of int, array-like of bool, slice or callable Indexes the data on its second axis. Integers are interpreted as positional columns, while strings can reference DataFrame columns by name. A scalar string or int should be used where ``transformer`` expects X to be a 1d array-like (vector), otherwise a 2d array will be passed to the transformer. A callable is passed the input data `X` and can return any of the above. To select multiple columns by name or dtype, you can use :obj:`make_column_selector`.","[('categorical', ...), ('numeric', ...)]"
,"remainder remainder: {'drop', 'passthrough'} or estimator, default='drop'By default, only the specified columns in `transformers` aretransformed and combined in the output, and the non-specifiedcolumns are dropped. (defaul

In [219]:
# Predict that target for the training rows and for the validation rows
predictions_train = tree.predict(X_train)
predictions_val = tree.predict(X_val)


In [220]:
# Check accuracy with .score() and f1_score, on the training rows and validation rows
untuned_train_accuracy = tree.score(X_train, y_train)
untuned_val_accuracy = tree.score(X_val, y_val)
untuned_train_f1 = f1_score(y_train, predictions_train)
untuned_val_f1 = f1_score(y_val, predictions_val)

print(f"Default tree  train: accuracy={untuned_train_accuracy:.3f}  F1={untuned_train_f1:.3f}")
print(f"Default tree  val:   accuracy={untuned_val_accuracy:.3f}  F1={untuned_val_f1:.3f}")

Default tree  train: accuracy=1.000  F1=1.000
Default tree  val:   accuracy=0.633  F1=0.675


In [221]:
# One tree per candidate depth. None means "grow until the leaves
# are pure", so it is the most flexible option in the list.
depth_results = []
for depth in [1, 2, 3, 4, 5, 6, 8, 10, None]:
    candidate =  make_pipeline(
        preprocessor,
        DecisionTreeClassifier(
            random_state=RANDOM_STATE, 
            max_depth=depth)
    )
    candidate.fit(X_train, y_train)
    # Train and validation for both metrics. The gap between the two columns
    # of a metric is what shows overfitting, so neither half is droppable.
    # F1 itself is defined properly later in the course.
    depth_results.append(
        {
            "max_depth": str(depth),
            "train_accuracy": candidate.score(X_train, y_train),
            "val_accuracy": candidate.score(X_val, y_val),
            "train_f1": f1_score(y_train, candidate.predict(X_train)),
            "val_f1": f1_score(y_val, candidate.predict(X_val)),
        }
    )

depth_table = pd.DataFrame(depth_results)
depth_table.round(3)

,max_depth,train_accuracy,val_accuracy,train_f1,val_f1
0,1,0.659,0.677,0.609,0.634
1,2,0.694,0.698,0.706,0.712
2,3,0.697,0.688,0.682,0.677
3,4,0.718,0.691,0.743,0.729
4,5,0.722,0.693,0.746,0.723
5,6,0.749,0.704,0.773,0.735
6,8,0.794,0.678,0.817,0.715
7,10,0.859,0.665,0.872,0.694
8,None,1.000,0.633,1.000,0.675


In [222]:
# What does the metric choice cost? Look at the top of the ranking.
print("Best two by validation accuracy:")
print(
    depth_table.nlargest(2, "val_accuracy")[
        ["max_depth", "train_accuracy", "val_accuracy", "train_f1", "val_f1"]
    ].round(3).to_string(index=False)
)
print(f"\nalways-predict-legal accuracy: {1 - y_val.mean():.3f}")

Best two by validation accuracy:
max_depth  train_accuracy  val_accuracy  train_f1  val_f1
        6           0.749         0.704     0.773   0.735
        2           0.694         0.698     0.706   0.712

always-predict-legal accuracy: 0.440


In [223]:
# Sort by score, then by depth, so a tie goes to the simpler tree
# instead of to whichever row pandas happened to put first.
ranked = depth_table.copy()
ranked["depth_for_sorting"] = ranked["max_depth"].replace("None", "999").astype(int)
best_row = ranked.sort_values(
    ["val_accuracy", "depth_for_sorting"],
    ascending=[False, True],
).iloc[0]
best_depth = None if best_row["max_depth"] == "None" else int(
    best_row["max_depth"]
)

tree_tuned = make_pipeline(
    preprocessor,
    DecisionTreeClassifier(
        max_depth=best_depth,
        random_state=RANDOM_STATE,
    )
)
tree_tuned.fit(X_train, y_train)
best_depth, round(float(best_row["val_accuracy"]), 3)

(6, 0.704)